In [1]:
%pip install duckduckgo-search


Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -U ddgs

Note: you may need to restart the kernel to use updated packages.


In [4]:
import langchain
import langchain_community
print(f"LangChain: {langchain.__version__}")
print(f"LangChain Community: {langchain_community.__version__}")

LangChain: 1.3.10
LangChain Community: 0.4.2


C:\Users\quick\AppData\Local\Temp\ipykernel_30156\909145161.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [5]:
# Imports

import re
import os

In [6]:
import os
from langchain_groq import ChatGroq

api_key = os.getenv("GROQ_API_KEY")
llm=ChatGroq(model_name="llama-3.1-8b-instant",groq_api_key=api_key,temperature=0,
             max_tokens=1024,)
# Quick sanity check
response = llm.invoke("Say hello in one sentence.")
print(response.content)

c:\Users\quick\anaconda3\envs\computer_vision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hello, how can I assist you today?


#### Defining tools

we will be using 2 tools
1. Search tool
2. Custom math tool


In [ ]:
import re
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

# Tool 1
search_tool=DuckDuckGoSearchRun()
search_tool.name="Search"
search_tool.description = (
    "Useful for searching current information on the internet. "
    "Use this when you need facts about recent events, people, "
    "places, or anything you are not certain about. "
    "Input should be a plain search query string."
)

# Tool 2

@tool
def math(expression: str) -> str:
    try:
        cleaned = re.sub(r"[^0-9+\-*/().%\s]", "", expression)
        result = eval(
            cleaned,
            {"__builtins__": None},
            {}
        )
        return str(result)

    except Exception as e:
        return f"Error: {e}"

In [8]:
print(math.invoke("2+3"))
print(math.invoke("2021+5"))

5
2026


In [9]:
tools = [search_tool,math]

#### STEP 5 — Pull ReAct prompt and create agent

1. Pull the standard ReAct prompt from LangChain hub.
2. This is the prompt that instructs the LLM to follow the
3. Thought → Action → Action Input → Observation format.

In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful AI assistant.

When solving a task:

- Think about what information you need.
- Use tools whenever necessary.
- Continue until you have enough information.
- Then return the final answer.
"""
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant."
)

In [11]:
from langchain_core.messages import AIMessage

def run_query(label: str, query: str):
    print(label)
    print(f"Query: {query}")

    try:
        result = agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": query,
                    }
                ]
            }
        )
        # Find the last AI response
        for message in reversed(result["messages"]):
            if isinstance(message, AIMessage):
                print(f"\nFinal Answer:\n{message.content}")
                break

        return result

    except Exception as e:
        print(f"\n Error: {e}")
        return None

In [12]:
r1 = run_query(
    "QUERY 1: Search only",
    "Who is the current CEO of Anthropic and when was the company founded?"
)

QUERY 1: Search only
Query: Who is the current CEO of Anthropic and when was the company founded?

Final Answer:
Dario Amodei was born in 1983.


In [13]:
r2 = run_query(
    "QUERY 2: Math only",
    "A model processes 512 tokens per second. "
    "I have a document with 8192 tokens. "
    "How many seconds will it take to process? What is that in minutes?"
)

QUERY 2: Math only
Query: A model processes 512 tokens per second. I have a document with 8192 tokens. How many seconds will it take to process? What is that in minutes?

 Error: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=math>{expression: 8192 / 512}</function>\n'}}


In [14]:
r3 = run_query(
    "QUERY 3: Search + Math combined",
    "What is the current population of India? "
    "If 40% of that population has access to the internet, "
    "how many people is that?"
)

QUERY 3: Search + Math combined
Query: What is the current population of India? If 40% of that population has access to the internet, how many people is that?

Final Answer:
Based on the information provided, the current population of India is approximately 1.476 billion, and 40% of that population has access to the internet, which is approximately 0.59 billion people.
